# Verify Minimal Architecure

## Load dim_backprop.py

In [ ]:
# %load dim_backprop.py
from sage.all import *


class Network(object):

    def __init__(self, sizes, exponent=2, finite_field=100003):
        """The list ``sizes`` contains the number of neurons in the
        respective layers of the network.  For example, if the list
        was [2, 3, 1] then it would be a three-layer network, with the
        first layer containing 2 neurons, the second layer 3 neurons,
        and the third layer 1 neuron.  Currently, biases are initialized to 
        zero and weights are initialized randomly."""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.FF = GF(finite_field)
        self.biases = [zero_matrix(self.FF, y, 1) for y in sizes[1:]]  # [random_matrix(FF, y, 1) for y in sizes[1:]]
        self.weights = [random_matrix(self.FF, y, x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.exponent = exponent
        self.degree = self.exponent**(self.num_layers - 2)

    def feedforward(self, a):
        """Return the output of the network if ``a`` is input."""
        for b, w in zip(self.biases[:-1], self.weights[:-1]):
            a = matrix_power(w * a + b, self.exponent)
        return self.weights[-1] * a + self.biases[-1]

    def backprop(self, x, pullback_vector=None):
        """Return a tuple ``(nabla_b, nabla_w)`` representing the
        gradient for the *output*pullback_vector* function.  ``nabla_b`` and
        ``nabla_w`` are layer-by-layer lists of matrices, similar
        to ``self.biases`` and ``self.weights``."""

        if pullback_vector is None:
            pullback_vector = ones_matrix(self.FF, self.sizes[-1], 1)

        nabla_b = [zero_matrix(self.FF, b.nrows(), b.ncols()) for b in self.biases]
        nabla_w = [zero_matrix(self.FF, w.nrows(), w.ncols()) for w in self.weights]
        # feedforward
        activation = x
        activations = [x]  # list to store all the activations, layer by layer
        zs = []  # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = w * activation + b
            zs.append(z)
            activation = matrix_power(z, self.exponent)
            activations.append(activation)
        # backward pass
        delta = pullback_vector
        nabla_b[-1] = delta
        nabla_w[-1] = delta * activations[-2].T
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = matrix_power_prime(z, self.exponent)
            delta = elementwise_product(self.weights[-l + 1].T * delta, sp)
            nabla_b[-l] = delta
            nabla_w[-l] = delta * activations[-l - 1].T
        return (nabla_b, nabla_w)


def matrix_power(M, exponent=2):
    """Raise elements in M to the power exponent."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c]**exponent
    return A


def matrix_power_prime(M, exponent=2):
    """Derivative of matrix_power."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = exponent * M[r, c]**(exponent - 1)
    return A


def elementwise_product(M, N):
    """Element-wise product of M and N."""
    nc, nr = M.ncols(), M.nrows()
    A = copy(M.parent().zero())
    for r in range(nr):
        for c in range(nc):
            A[r, c] = M[r, c] * N[r, c]
    return A


def monomial_list(v, k):
    """Return a list of all monomials in the entries of v of degree k."""
    nvars = len(v)
    exponents_list = list(WeightedIntegerVectors(k, [1 for t in v]))
    return [prod([v[i] ** exponents[i] for i in range(nvars)]) for exponents in exponents_list]


def compute_dimension(network_widths, network_exponent):
    """Compute the dimension of the neurovariety of a network with arbitrary output dimension."""

    primes = [100003, 100019]   # run the computation over GF(p) for p in primes
    dims = []
    for p in primes:
        nn = Network(network_widths, network_exponent, p)
        num_params = sum([m * n for m, n in zip(nn.sizes[:-1], nn.sizes[1:])])
        degree = nn.degree
        dim_poly_vector = binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)
        nsamples = 2 * dim_poly_vector  # nsamples should be at least dim_poly_vector
        x = random_matrix(nn.FF, nn.sizes[0], nsamples)
        monomials = matrix(nn.FF, [monomial_list(v, degree) for v in x.T])
        monomials_pinv = monomials.pseudoinverse()
        jacobian_matrix = zero_matrix(nn.FF, nn.sizes[-1] * dim_poly_vector, num_params)
        for j in range(nn.sizes[-1]):
            gradients_samples = zero_matrix(nn.FF, nsamples, num_params)
            basis_vec = zero_matrix(nn.FF, nn.sizes[-1], 1)
            basis_vec[j, 0] = 1
            for i in range(nsamples):
                gradient_matrices = nn.backprop(x[:, i], basis_vec)[1]  # use no biases
                gradients_samples[i, :] = matrix(nn.FF, [[t for mat in gradient_matrices for t in mat.list()]])  
            jacobian_matrix[j * dim_poly_vector:(j + 1) * dim_poly_vector, :] = monomials_pinv * gradients_samples
        dims.append(rank(jacobian_matrix))
    if not all(d == dims[0] for d in dims):
        raise ValueError('different dimensions over finite fields: ' + str(dims))
    ambient_dim = (binomial(nn.degree + nn.sizes[0] - 1, nn.sizes[0] - 1)) * nn.sizes[-1]
    naive_bound = sum([(m - 1) * n for m, n in zip(network_widths[:-1], network_widths[1:])]) + network_widths[-1]
    ex_dim = min(ambient_dim, naive_bound)
    return( nn.sizes, # d
            nn.exponent,                         # r
            ambient_dim,                         # ambient_dim
            ex_dim,                              # expected_dim
            dims[0],                           # dimension 
            ex_dim - dims[0]                   # defect
           )

In [ ]:
# %load is_unimodal.py
import ast

def is_unimodal(data):
    # 1. Parse the string into a list safely
    if isinstance(data, str):
        try:
            # ast.literal_eval safely evaluates strings containing Python literals
            data = ast.literal_eval(data)
        except (ValueError, SyntaxError):
            raise ValueError(f"Could not parse the string: '{data}'. Ensure it is formatted like '[1, 2, 3]'.")
            
    # 2. Validate the data type
    if not isinstance(data, list):
        raise TypeError("Input must be a list or a string representation of a list.")
        
    # 3. Core unimodal logic
    n = len(data)
    if n <= 2:
        return True
        
    i = 0
    
    # Phase 1: Walk up the non-decreasing slope
    while i + 1 < n and data[i] <= data[i + 1]:
        i += 1
        
    # Phase 2: Walk down the non-increasing slope
    while i + 1 < n and data[i] >= data[i + 1]:
        i += 1
        
    # Phase 3: Check if we reached the end
    return i == n - 1

## Functions to Verify Minimality

In [ ]:
def check_full_dimension_fast(sizes, exponent=2):
    """
    Computes the functional dimension by wrapping the existing compute_dimension 
    function from dim_backprop.py.
    
    Returns:
        tuple: (is_filling, actual_dimension, ambient_dimension, num_params)
    """
    # compute_dimension returns: (sizes, exponent, ambient_dim, expected_dim, actual_dim, defect)
    result = compute_dimension(sizes, exponent)
    
    ambient_dim = result[2]
    actual_dim = result[4]
    
    # Calculate num_params (since compute_dimension doesn't explicitly return it)
    num_params = sum([m * n for m, n in zip(sizes[:-1], sizes[1:])])
    
    # If parameters < ambient_dim, it trivially can't fill
    if num_params < ambient_dim:
        return False, actual_dim, ambient_dim, num_params
        
    # The architecture fills the space if its actual dimension equals the ambient dimension
    is_filling = (actual_dim == ambient_dim)
    
    return is_filling, actual_dim, ambient_dim, num_params

In [ ]:
def verify_minimality_brute_force(hidden_tuple, d_0, d_h, exponent=2):

    """
    Checks if a given architecture (d_0, hidden_tuple, d_h) is a minimal filling architecture
    by checking immediate subarchitectures (subtracting 1 from each coordinate).
    """

    target_sizes = [d_0] + list(hidden_tuple) + [d_h]
    print(f"="*100)
    print(f"EVALUATING TARGET ARCHITECTURE: {target_sizes}")
    print("="*100)
    
    # 1. Check if the target is FILLING
    is_full, dim, amb, params = check_full_dimension_fast(target_sizes, exponent)
    status = "FILLING" if is_full else "SHORT"
    print(f"\nTarget Result: [{status}] (Rank: {dim}/{amb}, Params: {params})")
    
    if not is_full:
        print("\n>>> CONCLUSION: NOT MINIMAL FILLING (Target itself does not fill the dimension) <<<")
        return False
    
    # 2. Generate immediate subarchitectures (subtract 1 from each coordinate)
    print("\nTarget is FILLING! Generating immediate smaller subsets to test:\n")
    
    immediate_subsets = []
    for i in range(len(hidden_tuple)):
        # Only subtract 1 if the layer width is greater than 1 
        # a width of 0 should break
        if hidden_tuple[i] > 1:
            subset = list(hidden_tuple)
            subset[i] -= 1
            immediate_subsets.append(tuple(subset))
    
    print(f"Total immediate subsets to verify: {len(immediate_subsets)}\n")
    
    # 3. Check the immediate predecessors
    for subset in immediate_subsets:
        pred_sizes = [d_0] + list(subset) + [d_h]
        
        pred_is_full, pred_dim, pred_amb, pred_params = check_full_dimension_fast(pred_sizes, exponent)
        
        p_status = "FILLING" if pred_is_full else "SHORT"
        print(f"  Testing Subset {pred_sizes} -> [{p_status}] (Jacobian Rank: {pred_dim}/{pred_amb}, Params: {pred_params})")
        
        if pred_is_full:
            print(f"\n>>> CONCLUSION: IT IS NOT A MINIMAL FILLING ARCHITECTURE <<<")
            print(f"A strictly smaller architecture {pred_sizes} is also FULL.")
            return False
            
    print(f"\n>>> CONCLUSION: CONFIRMED MINIMAL FILLING ARCHITECTURE <<<")
    print(f"The architecture is MINIMAL FILLING, and no immediate subarchitecture can fill the space.")
    return True

# Update .csv to Correctly Identify is_minimal Tuples

In [ ]:
import ast
import pandas as pd

INOUTS = [(2,1), 
          (2,2),
          (2,3)
          ]

for inoutwidths in INOUTS:
    d0=inoutwidths[0]
    dL=inoutwidths[1]
    
    for USER_DEPTH in range(1,9):

        # Keep the full dataframe
        df_full = pd.read_csv(f'../data/raw/{d0}_{dL}_architectures.csv')
        df_full['is_unimodal'] = df_full['architecture'].apply(is_unimodal)

        # Create mask of dataframe to check
        mask = (df_full['h'] == USER_DEPTH) & (df_full['is_minimal'] == True) 

        print("--"*10 + "UPDATING MINIMAL FILLING ARCHITECTURES" + "--"*10)


        for index, row in df_full[mask].iterrows():
            
            # 1. Setup the variables from the row
            arch = row['architecture']
            
            if isinstance(arch, str):
                arch = ast.literal_eval(arch)
                
            USER_D_0 = arch[0]
            USER_D_H = arch[-1]
            USER_GUESS = tuple(arch[1:-1])
            
            print(f"\nEvaluating Row {index}: Architecture {arch}")
            
            # 2. Verify
            is_minimal = verify_minimality_brute_force(
                hidden_tuple=USER_GUESS, 
                d_0=USER_D_0, 
                d_h=USER_D_H, 
                exponent=row['exponent'], 
            )
            
            # 3. Update the original dataframe if it fails the check
            if not is_minimal:
                df_full.at[index, 'is_minimal'] = False

            # Save the safely updated, complete dataset
            df_full.to_csv(f'../data/raw/{d0}_{dL}_architectures.csv', index=False)
            print(f"\nUpdated a row. Saved to '{d0}_{dL}_architectures.csv'")

# Verify Minimal Unimodal Example

To use the following code block, we load one of the .csv files produced from the search. One should filter for reasonable entries to check minimality (most will not be minimal).

In [ ]:
import ast
import pandas as pd

USER_DEPTH = 7
df = pd.read_csv('../data/raw/2_3_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*10 + "DISCOVERED NONUNIMODAL MINIMAL FILLING ARCHITECTURES" + "--"*10)
df = df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']== False)]

verified_results = []

for index, row in df.iterrows():
    arch = row['architecture']
    
    if isinstance(arch, str):
        arch = ast.literal_eval(arch)
        
    USER_D_0 = arch[0]
    USER_D_H = arch[-1]
    USER_GUESS = tuple(arch[1:-1])
    
    print(f"\nEvaluating Row {index}: Architecture {arch}")
    
    is_minimal = verify_minimality_brute_force(
        hidden_tuple=USER_GUESS, 
        d_0=USER_D_0, 
        d_h=USER_D_H, 
        exponent=row['exponent'], 
    )
    
    verified_results.append(is_minimal)